In [19]:
%pip install -q -r ../requirements.txt


Note: you may need to restart the kernel to use updated packages.


In [1]:
import os  
import dotenv
from openai import AzureOpenAI  

dotenv.load_dotenv(override=True)


True

In [2]:
user_input = "return a query that will include demographic information and visit information for a specific person"
user_input = "return a query that will include demographic information, visit and procedure information for a specific person"

In [ ]:
subscription_key = os.environ["AZURE_OPENAI_API_KEY"]
api_version = os.environ["AZURE_OPENAI_API_VERSION"]
endpoint = os.environ["AZURE_OPENAI_ENDPOINT"]
deployment = os.environ["AZURE_OPENAI_DEPLOYMENT"]
embedding_deployment = os.environ["AZURE_OPENAI_EMBEDDING_DEPLOYMENT"]
search_key = os.environ["AZURE_AI_SEARCH_KEY"]
search_endpoint = os.environ["AZURE_AI_SEARCH_ENDPOINT"]
search_index = os.environ["AZURE_AI_SEARCH_INDEX_NAME"]

embedding_endpoint = f"{endpoint}openai/deployments/{embedding_deployment}/embeddings?api-version={api_version}"


# Initialize Azure OpenAI client with key-based authentication
client = AzureOpenAI(  
    azure_endpoint=endpoint,  
    api_key=subscription_key,  
    api_version="2024-05-01-preview",  
)  
    
# Prepare the chat prompt  
chat_prompt = [
{
    "role": "system",
    "content": "You are an AI assistant that helps people write SQL queries"
},
{
    "role": "user",
    "content": user_input
}
]  


# Generate the completion  
completion = client.chat.completions.create(  
    model=deployment,  
    messages=chat_prompt,  
    max_tokens=900,  
    temperature=0.7,  
    top_p=0.95,  
    frequency_penalty=0,  
    presence_penalty=0,  
    stop=None,  
    stream=False  ,
    extra_body={
    "data_sources": [{
        "type": "azure_search",
        "parameters": {
        "filter": None,
        "endpoint": f"{search_endpoint}",
        "index_name": f"{search_index}",
        "semantic_configuration": "",
        "authentication": {
            "type": "api_key",
            "key": f"{search_key}"
        },
        "embedding_dependency": {
            "type": "endpoint",
            "endpoint": f"{embedding_endpoint}",
            "authentication": {
            "type": "api_key",
            "key": f"{subscription_key}"
            }
        },
        "query_type": "vector_simple_hybrid",
        "in_scope": True,
        "role_information": "You are an AI assistant that helps people write SQL queries",
        "strictness": 3,
        "top_n_documents": 5
        }
    }]
    }
)   
response = completion.to_json()
print(response)  


{
  "id": "61a9691f-7472-46cd-bea8-48af99ed554a",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "message": {
        "content": "To create a query that includes demographic information, visit, and procedure information for a specific person, you can use the following SQL query. This query assumes that you have tables for person, visit_occurrence, and procedure_occurrence. However, note that the procedure_occurrence table is not provided in the retrieved documents, so you may need to adjust the query according to your actual schema.\n\n```sql\nSELECT \n    p.person_id,\n    p.gender_concept_id,\n    p.year_of_birth,\n    p.month_of_birth,\n    p.day_of_birth,\n    p.race_concept_id,\n    p.ethnicity_concept_id,\n    v.visit_occurrence_id,\n    v.visit_concept_id,\n    v.visit_start_date,\n    v.visit_end_date,\n    po.procedure_occurrence_id,\n    po.procedure_concept_id,\n    po.procedure_date\nFROM \n    person p\nJOIN \n    visit_occurrence v ON p.person

In [4]:
print(completion.to_dict()['choices'][0]['message']['content'])

To create a query that includes demographic information, visit, and procedure information for a specific person, you can use the following SQL query. This query assumes that you have tables for person, visit_occurrence, and procedure_occurrence. However, note that the procedure_occurrence table is not provided in the retrieved documents, so you may need to adjust the query according to your actual schema.

```sql
SELECT 
    p.person_id,
    p.gender_concept_id,
    p.year_of_birth,
    p.month_of_birth,
    p.day_of_birth,
    p.race_concept_id,
    p.ethnicity_concept_id,
    v.visit_occurrence_id,
    v.visit_concept_id,
    v.visit_start_date,
    v.visit_end_date,
    po.procedure_occurrence_id,
    po.procedure_concept_id,
    po.procedure_date
FROM 
    person p
JOIN 
    visit_occurrence v ON p.person_id = v.person_id
JOIN 
    procedure_occurrence po ON p.person_id = po.person_id
WHERE 
    p.person_id = <SPECIFIC_PERSON_ID>;
```

Replace `<SPECIFIC_PERSON_ID>` with the actual